In [1]:
import arcpy
import os
import re
import time
from tqdm import tqdm

arcpy.env.overwriteOutput = True

arcpy.CheckOutExtension("GeoStats")
arcpy.CheckOutExtension("Spatial")

# --------------------------------------------------
# INPUTS
# --------------------------------------------------
geochem_points = r"C:\Users\acosta_pedro\OneDrive - Norges geologiske undersøkelse\Geochemistry NGU_2025\kalk_prosjekt3.0\Geochemistry\shape\All_berggrunns_XRFH_klassifisert_nearpoint.shp"

template_mask = r"E:\mask\Mask_land_Kalk_cog.tif"

out_folder = r"G:\Geology_rasters_2026\Geochemistry\EBK"
os.makedirs(out_folder, exist_ok=True)

z_fields = ["CaO", "CaMg_LOI_v", "Comp_1"]

# --------------------------------------------------
# RESOLUTION SETTINGS
# --------------------------------------------------
# EBK is performed at 250 m, then resampled/aligned to the 10 m mask.
EBK_CELL_SIZE = 250

# --------------------------------------------------
# CHECK INPUTS
# --------------------------------------------------
for path in [geochem_points, template_mask]:
    if not arcpy.Exists(path):
        raise Exception(f"Input not found:\n{path}")

point_fields = [f.name for f in arcpy.ListFields(geochem_points)]

for fld in z_fields:
    if fld not in point_fields:
        raise Exception(f"Field '{fld}' not found in point shapefile.")

# --------------------------------------------------
# TEMPLATE RASTER PROPERTIES
# --------------------------------------------------
template_desc = arcpy.Describe(template_mask)
template_sr = template_desc.spatialReference

mask_cell_size_x = float(arcpy.management.GetRasterProperties(template_mask, "CELLSIZEX")[0])
mask_cell_size_y = float(arcpy.management.GetRasterProperties(template_mask, "CELLSIZEY")[0])

print("Template raster:")
print(template_mask)
print(f"CRS      : {template_sr.name}")
print(f"Cell size: {mask_cell_size_x} x {mask_cell_size_y}")
print(f"Rows     : {arcpy.management.GetRasterProperties(template_mask, 'ROWCOUNT')[0]}")
print(f"Cols     : {arcpy.management.GetRasterProperties(template_mask, 'COLUMNCOUNT')[0]}")

print("\nEBK interpolation cell size:")
print(f"{EBK_CELL_SIZE} m")

# --------------------------------------------------
# SCRATCH
# --------------------------------------------------
scratch_gdb = arcpy.env.scratchGDB

print("\nScratch geodatabase:")
print(scratch_gdb)

# --------------------------------------------------
# PROJECT POINTS TO TEMPLATE CRS
# --------------------------------------------------
geochem_projected = os.path.join(scratch_gdb, "geochem_points_projected_utm33")

if arcpy.Exists(geochem_projected):
    arcpy.management.Delete(geochem_projected)

point_sr = arcpy.Describe(geochem_points).spatialReference

print("\nPoint CRS:")
print(point_sr.name)

if point_sr.factoryCode != template_sr.factoryCode:
    print("Projecting points to template raster CRS...")

    arcpy.management.Project(
        in_dataset=geochem_points,
        out_dataset=geochem_projected,
        out_coor_system=template_sr
    )

    geochem_for_ebk = geochem_projected
    print("Projected points created.")
else:
    geochem_for_ebk = geochem_points
    print("Points already match template CRS.")

# --------------------------------------------------
# HELPER FUNCTION
# --------------------------------------------------
def safe_name(name):
    name = re.sub(r"[^A-Za-z0-9_]+", "_", name)
    return name[:40]

# --------------------------------------------------
# EBK INTERPOLATION WITH PROGRESS BAR + ETA
# --------------------------------------------------
start_all = time.time()

with tqdm(total=len(z_fields), desc="EBK interpolation", unit="variable") as pbar:

    for i, fld in enumerate(z_fields, start=1):

        var_start = time.time()

        print("\n" + "=" * 70)
        print(f"Processing EBK for: {fld} ({i}/{len(z_fields)})")
        print("=" * 70)

        fld_safe = safe_name(fld)

        point_lyr = f"pts_{fld_safe}"

        ga_layer = os.path.join(scratch_gdb, f"EBK_ga_{fld_safe}")

        # EBK raster at 250 m
        tmp_ebk_raster = os.path.join(scratch_gdb, f"EBK_250m_{fld_safe}")

        # Resampled raster aligned to 10 m template
        tmp_resampled = os.path.join(scratch_gdb, f"EBK_10m_resamp_{fld_safe}")

        # Final masked/aligned raster before COG
        tmp_masked = os.path.join(scratch_gdb, f"EBK_10m_masked_{fld_safe}")

        # Final COG
        out_cog = os.path.join(out_folder, f"EBK_{fld_safe}_10m_aligned.tif")

        # --------------------------------------------------
        # CLEAN TEMPORARY OUTPUTS
        # --------------------------------------------------
        for p in [point_lyr, ga_layer, tmp_ebk_raster, tmp_resampled, tmp_masked]:
            try:
                if arcpy.Exists(p):
                    arcpy.management.Delete(p)
            except Exception:
                pass

        if arcpy.Exists(out_cog):
            arcpy.management.Delete(out_cog)

        # --------------------------------------------------
        # SELECT VALID NON-NULL VALUES
        # --------------------------------------------------
        where_clause = f"{arcpy.AddFieldDelimiters(geochem_for_ebk, fld)} IS NOT NULL"

        arcpy.management.MakeFeatureLayer(
            in_features=geochem_for_ebk,
            out_layer=point_lyr,
            where_clause=where_clause
        )

        n_valid = int(arcpy.management.GetCount(point_lyr)[0])
        print(f"Valid points for {fld}: {n_valid}")

        if n_valid < 30:
            print(f"Skipping {fld}: too few valid points for reliable EBK.")
            pbar.update(1)
            continue

        # --------------------------------------------------
        # ENVIRONMENT FOR EBK AT 250 m
        # --------------------------------------------------
        arcpy.env.snapRaster = None
        arcpy.env.cellSize = EBK_CELL_SIZE
        arcpy.env.extent = template_mask
        arcpy.env.outputCoordinateSystem = template_sr
        arcpy.env.mask = None

        # --------------------------------------------------
        # EMPIRICAL BAYESIAN KRIGING AT 250 m
        # --------------------------------------------------
        arcpy.ga.EmpiricalBayesianKriging(
            in_features=point_lyr,
            z_field=fld,
            out_ga_layer=ga_layer,
            out_raster=tmp_ebk_raster,
            cell_size=EBK_CELL_SIZE,
            transformation_type="EMPIRICAL",
            max_local_points=100,
            overlap_factor=1,
            number_semivariograms=100,
            search_neighborhood="NBRTYPE=StandardCircular RADIUS=50000 ANGLE=0 NBR_MAX=15 NBR_MIN=10 SECTOR_TYPE=ONE_SECTOR",
            output_type="PREDICTION",
            semivariogram_model_type="K_BESSEL"
        )

        print("EBK raster created at 250 m.")

        # --------------------------------------------------
        # ENVIRONMENT FOR RESAMPLING TO TEMPLATE GRID
        # --------------------------------------------------
        arcpy.env.snapRaster = template_mask
        arcpy.env.cellSize = template_mask
        arcpy.env.extent = template_mask
        arcpy.env.outputCoordinateSystem = template_sr
        arcpy.env.mask = None

        # --------------------------------------------------
        # RESAMPLE EBK RASTER TO 10 m TEMPLATE GRID
        # --------------------------------------------------
        arcpy.management.Resample(
            in_raster=tmp_ebk_raster,
            out_raster=tmp_resampled,
            cell_size=f"{mask_cell_size_x} {mask_cell_size_y}",
            resampling_type="BILINEAR"
        )

        print("Resampled to template cell size.")

        # --------------------------------------------------
        # MASK TO TEMPLATE LAND MASK
        # --------------------------------------------------
        masked_raster = arcpy.sa.ExtractByMask(
            in_raster=tmp_resampled,
            in_mask_data=template_mask
        )

        masked_raster.save(tmp_masked)

        print("Masked to template raster.")

        # --------------------------------------------------
        # SAVE FINAL ALIGNED CLOUD OPTIMIZED GEOTIFF
        # --------------------------------------------------
        arcpy.management.CopyRaster(
            in_raster=tmp_masked,
            out_rasterdataset=out_cog,
            pixel_type="32_BIT_FLOAT",
            format="COG"
        )

        # --------------------------------------------------
        # VERIFY FINAL RASTER ALIGNMENT
        # --------------------------------------------------
        out_desc = arcpy.Describe(out_cog)

        out_rows = int(arcpy.management.GetRasterProperties(out_cog, "ROWCOUNT")[0])
        out_cols = int(arcpy.management.GetRasterProperties(out_cog, "COLUMNCOUNT")[0])
        mask_rows = int(arcpy.management.GetRasterProperties(template_mask, "ROWCOUNT")[0])
        mask_cols = int(arcpy.management.GetRasterProperties(template_mask, "COLUMNCOUNT")[0])

        out_cell_x = float(arcpy.management.GetRasterProperties(out_cog, "CELLSIZEX")[0])
        out_cell_y = float(arcpy.management.GetRasterProperties(out_cog, "CELLSIZEY")[0])

        print("\nAlignment check:")
        print(f"Rows match: {out_rows == mask_rows} ({out_rows} vs {mask_rows})")
        print(f"Cols match: {out_cols == mask_cols} ({out_cols} vs {mask_cols})")
        print(f"Cell X match: {abs(out_cell_x - mask_cell_size_x) < 1e-9} ({out_cell_x})")
        print(f"Cell Y match: {abs(out_cell_y - mask_cell_size_y) < 1e-9} ({out_cell_y})")
        print(f"CRS match: {out_desc.spatialReference.factoryCode == template_sr.factoryCode}")

        var_minutes = (time.time() - var_start) / 60
        elapsed_minutes = (time.time() - start_all) / 60
        avg_minutes = elapsed_minutes / i
        remaining_minutes = avg_minutes * (len(z_fields) - i)

        print(f"\nSaved final aligned COG: {out_cog}")
        print(f"Time for {fld}: {var_minutes:.1f} minutes")
        print(f"Estimated remaining time: {remaining_minutes:.1f} minutes")

        pbar.set_postfix({
            "last_min": f"{var_minutes:.1f}",
            "ETA_min": f"{remaining_minutes:.1f}"
        })

        pbar.update(1)

# --------------------------------------------------
# RESET ENVIRONMENT
# --------------------------------------------------
arcpy.env.snapRaster = None
arcpy.env.cellSize = None
arcpy.env.extent = None
arcpy.env.outputCoordinateSystem = None
arcpy.env.mask = None

arcpy.CheckInExtension("GeoStats")
arcpy.CheckInExtension("Spatial")

total_minutes = (time.time() - start_all) / 60

print("\nFinished.")
print(f"Total runtime: {total_minutes:.1f} minutes")
print("Final EBK COG rasters saved in:")
print(out_folder)

Template raster:
E:\mask\Mask_land_Kalk_cog.tif
CRS      : ETRS_1989_UTM_Zone_33N
Cell size: 10.0 x 10.0
Rows     : 162232
Cols     : 135236

EBK interpolation cell size:
250 m

Scratch geodatabase:
C:\Users\acosta_pedro\OneDrive - Norges geologiske undersøkelse\Geochemistry NGU_2025\kalk_prosjekt3.0\ArcProject\Variabler_nasjonal_skala2025\scratch.gdb

Point CRS:
GCS_WGS_1984
Projecting points to template raster CRS...
Projected points created.


EBK interpolation:   0%|          | 0/3 [00:00<?, ?variable/s]


Processing EBK for: CaO (1/3)
Valid points for CaO: 21007
EBK raster created at 250 m.
Resampled to template cell size.
Masked to template raster.

Alignment check:
Rows match: True (162232 vs 162232)
Cols match: True (135236 vs 135236)
Cell X match: True (10.0)
Cell Y match: True (10.0)
CRS match: True

Saved final aligned COG: G:\Geology_rasters_2026\Geochemistry\EBK\EBK_CaO_10m_aligned.tif
Time for CaO: 156.5 minutes
Estimated remaining time: 313.1 minutes


EBK interpolation:  33%|███▎      | 1/3 [2:36:32<5:13:05, 9392.71s/variable, last_min=156.5, ETA_min=313.1]


Processing EBK for: CaMg_LOI_v (2/3)
Valid points for CaMg_LOI_v: 21007
EBK raster created at 250 m.
Resampled to template cell size.


EBK interpolation:  33%|███▎      | 1/3 [5:25:55<10:51:50, 19555.37s/variable, last_min=156.5, ETA_min=313.1]


<class 'UnicodeDecodeError'>: 'utf-8' codec can't decode byte 0xf8 in position 104: invalid start byte

In [1]:
import arcpy
import os
import re
import time
from tqdm import tqdm

arcpy.env.overwriteOutput = True

arcpy.CheckOutExtension("GeoStats")
arcpy.CheckOutExtension("Spatial")

# --------------------------------------------------
# INPUTS
# --------------------------------------------------
geochem_points = r"C:\Users\acosta_pedro\OneDrive - Norges geologiske undersøkelse\Geochemistry NGU_2025\kalk_prosjekt3.0\Geochemistry\shape\All_berggrunns_XRFH_klassifisert_nearpoint.shp"

template_mask = r"E:\mask\Mask_land_Kalk_cog.tif"

out_folder = r"G:\Geology_rasters_2026\Geochemistry\EBK"
os.makedirs(out_folder, exist_ok=True)

z_fields = ["CaO", "CaMg_LOI_v", "Comp_1"]

# --------------------------------------------------
# USER SETTINGS
# --------------------------------------------------
EBK_CELL_SIZE = 250  # meters

# If True, existing final COGs are skipped
SKIP_EXISTING_OUTPUTS = True

# If True, existing final COGs are deleted and recalculated
# Only used when SKIP_EXISTING_OUTPUTS = False
OVERWRITE_EXISTING_OUTPUTS = False

# Local temporary folder without special characters
local_tmp_dir = r"C:\temp\EBK"
os.makedirs(local_tmp_dir, exist_ok=True)

local_gdb = os.path.join(local_tmp_dir, "EBK_temp.gdb")

if not arcpy.Exists(local_gdb):
    arcpy.management.CreateFileGDB(local_tmp_dir, "EBK_temp.gdb")

scratch_gdb = local_gdb
arcpy.env.scratchWorkspace = scratch_gdb
arcpy.env.workspace = scratch_gdb

# --------------------------------------------------
# CHECK INPUTS
# --------------------------------------------------
for path in [geochem_points, template_mask]:
    if not arcpy.Exists(path):
        raise Exception(f"Input not found:\n{path}")

point_fields = [f.name for f in arcpy.ListFields(geochem_points)]

for fld in z_fields:
    if fld not in point_fields:
        raise Exception(f"Field '{fld}' not found in point shapefile.")

# --------------------------------------------------
# TEMPLATE RASTER PROPERTIES
# --------------------------------------------------
template_desc = arcpy.Describe(template_mask)
template_sr = template_desc.spatialReference

mask_cell_size_x = float(arcpy.management.GetRasterProperties(template_mask, "CELLSIZEX")[0])
mask_cell_size_y = float(arcpy.management.GetRasterProperties(template_mask, "CELLSIZEY")[0])

mask_rows = int(arcpy.management.GetRasterProperties(template_mask, "ROWCOUNT")[0])
mask_cols = int(arcpy.management.GetRasterProperties(template_mask, "COLUMNCOUNT")[0])

print("Template raster:")
print(template_mask)
print(f"CRS      : {template_sr.name}")
print(f"Cell size: {mask_cell_size_x} x {mask_cell_size_y}")
print(f"Rows     : {mask_rows}")
print(f"Cols     : {mask_cols}")

print("\nEBK interpolation cell size:")
print(f"{EBK_CELL_SIZE} m")

print("\nSkip existing outputs:")
print(SKIP_EXISTING_OUTPUTS)

print("\nOverwrite existing outputs:")
print(OVERWRITE_EXISTING_OUTPUTS)

print("\nScratch geodatabase:")
print(scratch_gdb)

print("\nTemporary raster folder:")
print(local_tmp_dir)

# --------------------------------------------------
# PROJECT POINTS TO TEMPLATE CRS
# --------------------------------------------------
geochem_projected = os.path.join(scratch_gdb, "geochem_points_projected_utm33")

point_sr = arcpy.Describe(geochem_points).spatialReference

print("\nPoint CRS:")
print(point_sr.name)

if point_sr.factoryCode != template_sr.factoryCode:

    if arcpy.Exists(geochem_projected):
        print("Projected points already exist in local scratch GDB.")
    else:
        print("Projecting points to template raster CRS...")

        arcpy.management.Project(
            in_dataset=geochem_points,
            out_dataset=geochem_projected,
            out_coor_system=template_sr
        )

        print("Projected points created.")

    geochem_for_ebk = geochem_projected

else:
    geochem_for_ebk = geochem_points
    print("Points already match template CRS.")

# --------------------------------------------------
# HELPER FUNCTIONS
# --------------------------------------------------
def safe_name(name):
    name = re.sub(r"[^A-Za-z0-9_]+", "_", name)
    return name[:40]


def delete_if_exists(path):
    try:
        if arcpy.Exists(path):
            arcpy.management.Delete(path)
    except Exception:
        pass


def check_alignment(raster_path, template_path):
    out_desc = arcpy.Describe(raster_path)
    template_desc_local = arcpy.Describe(template_path)

    out_rows = int(arcpy.management.GetRasterProperties(raster_path, "ROWCOUNT")[0])
    out_cols = int(arcpy.management.GetRasterProperties(raster_path, "COLUMNCOUNT")[0])

    tmp_rows = int(arcpy.management.GetRasterProperties(template_path, "ROWCOUNT")[0])
    tmp_cols = int(arcpy.management.GetRasterProperties(template_path, "COLUMNCOUNT")[0])

    out_cell_x = float(arcpy.management.GetRasterProperties(raster_path, "CELLSIZEX")[0])
    out_cell_y = float(arcpy.management.GetRasterProperties(raster_path, "CELLSIZEY")[0])

    tmp_cell_x = float(arcpy.management.GetRasterProperties(template_path, "CELLSIZEX")[0])
    tmp_cell_y = float(arcpy.management.GetRasterProperties(template_path, "CELLSIZEY")[0])

    rows_ok = out_rows == tmp_rows
    cols_ok = out_cols == tmp_cols
    cellx_ok = abs(out_cell_x - tmp_cell_x) < 1e-9
    celly_ok = abs(out_cell_y - tmp_cell_y) < 1e-9
    crs_ok = out_desc.spatialReference.factoryCode == template_desc_local.spatialReference.factoryCode

    print("\nAlignment check:")
    print(f"Rows match: {rows_ok} ({out_rows} vs {tmp_rows})")
    print(f"Cols match: {cols_ok} ({out_cols} vs {tmp_cols})")
    print(f"Cell X match: {cellx_ok} ({out_cell_x})")
    print(f"Cell Y match: {celly_ok} ({out_cell_y})")
    print(f"CRS match: {crs_ok}")

    return rows_ok and cols_ok and cellx_ok and celly_ok and crs_ok


# --------------------------------------------------
# EBK INTERPOLATION WITH PROGRESS BAR + ETA
# --------------------------------------------------
start_all = time.time()

with tqdm(total=len(z_fields), desc="EBK interpolation", unit="variable") as pbar:

    for i, fld in enumerate(z_fields, start=1):

        var_start = time.time()
        fld_safe = safe_name(fld)

        out_cog = os.path.join(out_folder, f"EBK_{fld_safe}_10m_aligned.tif")

        print("\n" + "=" * 70)
        print(f"Processing EBK for: {fld} ({i}/{len(z_fields)})")
        print("=" * 70)

        # --------------------------------------------------
        # SKIP / OVERWRITE EXISTING FINAL OUTPUT
        # --------------------------------------------------
        if arcpy.Exists(out_cog):

            if SKIP_EXISTING_OUTPUTS:
                print("Final COG already exists. Skipping.")
                print(out_cog)

                try:
                    check_alignment(out_cog, template_mask)
                except Exception:
                    print("Could not verify existing COG alignment, but file exists and was skipped.")

                pbar.update(1)
                continue

            elif OVERWRITE_EXISTING_OUTPUTS:
                print("Existing final COG found. Deleting.")
                delete_if_exists(out_cog)

            else:
                raise Exception(
                    f"Output already exists and both SKIP_EXISTING_OUTPUTS and "
                    f"OVERWRITE_EXISTING_OUTPUTS are False:\n{out_cog}"
                )

        # --------------------------------------------------
        # TEMPORARY OUTPUTS
        # --------------------------------------------------
        point_lyr = f"pts_{fld_safe}"

        ga_layer = os.path.join(scratch_gdb, f"EBK_ga_{fld_safe}")

        tmp_ebk_raster = os.path.join(
            scratch_gdb,
            f"EBK_250m_{fld_safe}"
        )

        # Important: temporary 10 m raster is a TIFF, not a GDB raster
        tmp_resampled_tif = os.path.join(
            local_tmp_dir,
            f"EBK_10m_{fld_safe}.tif"
        )

        # --------------------------------------------------
        # CLEAN TEMPORARY OUTPUTS FOR THIS VARIABLE
        # --------------------------------------------------
        for p in [
            point_lyr,
            ga_layer,
            tmp_ebk_raster,
            tmp_resampled_tif
        ]:
            delete_if_exists(p)

        # --------------------------------------------------
        # SELECT VALID NON-NULL VALUES
        # --------------------------------------------------
        where_clause = f"{arcpy.AddFieldDelimiters(geochem_for_ebk, fld)} IS NOT NULL"

        arcpy.management.MakeFeatureLayer(
            in_features=geochem_for_ebk,
            out_layer=point_lyr,
            where_clause=where_clause
        )

        n_valid = int(arcpy.management.GetCount(point_lyr)[0])
        print(f"Valid points for {fld}: {n_valid}")

        if n_valid < 30:
            print(f"Skipping {fld}: too few valid points for reliable EBK.")
            pbar.update(1)
            continue

        # --------------------------------------------------
        # ENVIRONMENT FOR EBK AT 250 m
        # --------------------------------------------------
        arcpy.env.snapRaster = None
        arcpy.env.cellSize = EBK_CELL_SIZE
        arcpy.env.extent = template_mask
        arcpy.env.outputCoordinateSystem = template_sr
        arcpy.env.mask = None

        # --------------------------------------------------
        # EMPIRICAL BAYESIAN KRIGING AT 250 m
        # --------------------------------------------------
        arcpy.ga.EmpiricalBayesianKriging(
            in_features=point_lyr,
            z_field=fld,
            out_ga_layer=ga_layer,
            out_raster=tmp_ebk_raster,
            cell_size=EBK_CELL_SIZE,
            transformation_type="EMPIRICAL",
            max_local_points=100,
            overlap_factor=1,
            number_semivariograms=100,
            search_neighborhood="NBRTYPE=StandardCircular RADIUS=50000 ANGLE=0 NBR_MAX=15 NBR_MIN=10 SECTOR_TYPE=ONE_SECTOR",
            output_type="PREDICTION",
            semivariogram_model_type="K_BESSEL"
        )

        print("EBK raster created at 250 m.")

        # --------------------------------------------------
        # RESAMPLE TO 10 m TEMPLATE GRID AS TEMPORARY TIFF
        # --------------------------------------------------
        arcpy.env.snapRaster = template_mask
        arcpy.env.cellSize = template_mask
        arcpy.env.extent = template_mask
        arcpy.env.outputCoordinateSystem = template_sr
        arcpy.env.mask = None

        arcpy.management.Resample(
            in_raster=tmp_ebk_raster,
            out_raster=tmp_resampled_tif,
            cell_size=f"{mask_cell_size_x} {mask_cell_size_y}",
            resampling_type="BILINEAR"
        )

        print("Resampled to 10 m template grid as temporary GeoTIFF.")

        # --------------------------------------------------
        # SAVE FINAL CLOUD OPTIMIZED GEOTIFF
        # --------------------------------------------------
        arcpy.management.CopyRaster(
            in_raster=tmp_resampled_tif,
            out_rasterdataset=out_cog,
            pixel_type="32_BIT_FLOAT",
            format="COG"
        )

        print("Saved final COG.")

        # --------------------------------------------------
        # VERIFY FINAL RASTER ALIGNMENT
        # --------------------------------------------------
        check_alignment(out_cog, template_mask)

        # --------------------------------------------------
        # OPTIONAL CLEANUP OF TEMPORARY 10 m TIFF
        # --------------------------------------------------
        delete_if_exists(tmp_resampled_tif)

        # --------------------------------------------------
        # TIMING
        # --------------------------------------------------
        var_minutes = (time.time() - var_start) / 60
        elapsed_minutes = (time.time() - start_all) / 60
        avg_minutes = elapsed_minutes / i
        remaining_minutes = avg_minutes * (len(z_fields) - i)

        print(f"\nSaved final aligned COG: {out_cog}")
        print(f"Time for {fld}: {var_minutes:.1f} minutes")
        print(f"Estimated remaining time: {remaining_minutes:.1f} minutes")

        pbar.set_postfix({
            "last_min": f"{var_minutes:.1f}",
            "ETA_min": f"{remaining_minutes:.1f}"
        })

        pbar.update(1)

# --------------------------------------------------
# RESET ENVIRONMENT
# --------------------------------------------------
arcpy.env.snapRaster = None
arcpy.env.cellSize = None
arcpy.env.extent = None
arcpy.env.outputCoordinateSystem = None
arcpy.env.mask = None
arcpy.env.workspace = None
arcpy.env.scratchWorkspace = None

arcpy.CheckInExtension("GeoStats")
arcpy.CheckInExtension("Spatial")

total_minutes = (time.time() - start_all) / 60

print("\nFinished.")
print(f"Total runtime: {total_minutes:.1f} minutes")
print("Final EBK COG rasters saved in:")
print(out_folder)

Template raster:
E:\mask\Mask_land_Kalk_cog.tif
CRS      : ETRS_1989_UTM_Zone_33N
Cell size: 10.0 x 10.0
Rows     : 162232
Cols     : 135236

EBK interpolation cell size:
250 m

Skip existing outputs:
True

Overwrite existing outputs:
False

Scratch geodatabase:
C:\temp\EBK\EBK_temp.gdb

Temporary raster folder:
C:\temp\EBK

Point CRS:
GCS_WGS_1984
Projected points already exist in local scratch GDB.


EBK interpolation:   0%|          | 0/3 [00:00<?, ?variable/s]


Processing EBK for: CaO (1/3)
Final COG already exists. Skipping.
G:\Geology_rasters_2026\Geochemistry\EBK\EBK_CaO_10m_aligned.tif

Alignment check:
Rows match: True (162232 vs 162232)
Cols match: True (135236 vs 135236)
Cell X match: True (10.0)
Cell Y match: True (10.0)
CRS match: True


EBK interpolation:  33%|███▎      | 1/3 [00:00<00:01,  1.94variable/s]


Processing EBK for: CaMg_LOI_v (2/3)
Valid points for CaMg_LOI_v: 21007
EBK raster created at 250 m.
Resampled to 10 m template grid as temporary GeoTIFF.
Saved final COG.

Alignment check:
Rows match: True (162232 vs 162232)
Cols match: True (135236 vs 135236)
Cell X match: True (10.0)
Cell Y match: True (10.0)
CRS match: True

Saved final aligned COG: G:\Geology_rasters_2026\Geochemistry\EBK\EBK_CaMg_LOI_v_10m_aligned.tif
Time for CaMg_LOI_v: 415.3 minutes
Estimated remaining time: 207.7 minutes


EBK interpolation:  67%|██████▋   | 2/3 [6:55:20<4:04:18, 14658.80s/variable, last_min=415.3, ETA_min=207.7]


Processing EBK for: Comp_1 (3/3)
Valid points for Comp_1: 21007
EBK raster created at 250 m.
Resampled to 10 m template grid as temporary GeoTIFF.
Saved final COG.

Alignment check:
Rows match: True (162232 vs 162232)
Cols match: True (135236 vs 135236)
Cell X match: True (10.0)
Cell Y match: True (10.0)
CRS match: True

Saved final aligned COG: G:\Geology_rasters_2026\Geochemistry\EBK\EBK_Comp_1_10m_aligned.tif
Time for Comp_1: 271.5 minutes
Estimated remaining time: 0.0 minutes


EBK interpolation: 100%|██████████| 3/3 [11:26:51<00:00, 13737.30s/variable, last_min=271.5, ETA_min=0.0]   ﻿



Finished.
Total runtime: 686.9 minutes
Final EBK COG rasters saved in:
G:\Geology_rasters_2026\Geochemistry\EBK
